# Part 2b — Benders / L-shaped decomposition

### Exploiting the LP core that every notebook since Part 3 has preserved

Part 2 solved the stochastic problem two ways: an extensive form (exact, but the whole tree at once)
and progressive hedging (scenario decomposition, a *heuristic* once the first stage has integers).
Neither exploited the structural fact the series has protected throughout — **the second stage is a
pure LP**.

That is precisely the condition under which the L-shaped method is exact:

| | Progressive hedging | **L-shaped / Benders** |
|---|---|---|
| Decomposes by | scenario | **stage** |
| First-stage integers | heuristic, no bound | **exact**, valid lower bound throughout |
| Needs | proximal parameter $\rho$, swept | nothing tuned |
| Second stage must be | anything | **convex with usable duals** |
| Gives you | a good solution | a solution **and** a certificate |

Progressive hedging with an integer first stage cannot tell you how far from optimal you are.
L-shaped can, because the master is a relaxation and its objective is a genuine lower bound at every
iteration.

### The mechanism in one line

The master keeps a variable $\theta_k$ standing in for scenario $k$'s recourse cost. Each solve
produces a candidate capacity plan; each subproblem returns its true cost **and a supporting
hyperplane** from the duals. Those cuts accumulate until $\theta$ stops lying.

$$\theta_k \;\ge\; Q_k(\hat c) \;+\; \sum_{n} \beta^k_n\,(c_n - \hat c_n)$$

$\beta^k_n$ is the dual on *capacity limits throughput* at node $n$ in scenario $k$ — the marginal
value of another unit of capacity there. That is the only piece of information the master needs.

### Style note

Multicut, not single-cut: one $\theta_k$ per scenario. It converges in far fewer iterations and the
cuts are individually interpretable. Single-cut is one line's change, shown in §7.

## Formulation reference

### First stage — here and now

$$\min_{y,c}\;\; \sum_n \big(F_n y_n + U_n c_n\big) \;+\; \sum_k p_k\,\theta_k$$
$$\underline{c}\,y_n \le c_n \le \overline{c}\,y_n, \qquad y_n \in \{0,1\}$$

### Second stage — scenario $k$, given $\hat c$

$$Q_k(\hat c) \;=\; \min_{x,f,u}\;\; \sum_n o_n x_n + \sum_a \tau_a f_a + \pi \sum_r u_r$$

subject to

| Constraint | | Dual |
|---|---|---|
| capacity | $x_n \le \hat c_n$ | $\beta_n \le 0$ |
| node output | $\eta_n x_n = \sum_a f_a$ | free |
| node input | $\sum_a f_a = x_n$ | free |
| demand | $\sum_a f_a + u_r \ge D_{r,k}$ | $\ge 0$ |

**Only $\beta$ carries first-stage information**, because $\hat c$ appears in exactly one place.
That is what makes the cut so cheap.

### The optimality cut

$$\theta_k \;\ge\; Q_k(\hat c) + \sum_n \beta^k_n (c_n - \hat c_n)$$

Valid because $Q_k$ is convex in $\hat c$ (an LP value function of its right-hand side) and $\beta$
is a subgradient.

### Why no feasibility cuts

The shortfall variable $u_r$ is unbounded above and penalised, so the subproblem is feasible for
**any** capacity plan including all-zero. Complete recourse — feasibility cuts never arise. This is
a modelling choice worth making deliberately: hard demand constraints would require them.

## 1. Setup

In [ ]:
import os, math, time
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt

for cand in ("gurobi.lic", os.path.join("..", "gurobi.lic")):
    if os.path.exists(cand):
        os.environ["GRB_LICENSE_FILE"] = os.path.abspath(cand); break

plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
print("gurobi", gp.gurobi.version())


## 2. Instance

Three stages, two regions, and a scenario set that differs in **demand**. Six capacity nodes, so
each cut has six coefficients.

In [ ]:
STAGES = ["MINE", "PROC", "MFG"]
REGS   = ["R1", "R2"]
NODES  = [(s, r) for s in STAGES for r in REGS]
ARCS   = [(s, r1, r2) for s in STAGES for r1 in REGS for r2 in REGS]

FIX  = {"MINE": 240., "PROC": 200., "MFG": 160.}
UNIT = {"MINE": 2.2,  "PROC": 2.9,  "MFG": 3.3}
OPC  = {"MINE": 0.8,  "PROC": 1.3,  "MFG": 1.6}
ETA  = {"MINE": 0.95, "PROC": 0.90, "MFG": 0.93}
CMIN, CMAX = 5.0, 70.0
TAU  = lambda r1, r2: 0.3 if r1 == r2 else 1.5
PEN  = 30.0

# scenarios: demand states
import itertools, random
random.seed(11)
NK = 24
SC = []
for k in range(NK):
    lo, hi = 0.55, 1.55
    a = lo + (hi-lo)*random.random()
    b = lo + (hi-lo)*random.random()
    SC.append({"R1": 34.0*a, "R2": 22.0*b})
PK = [1.0/NK]*NK
print(f"{NK} scenarios | {len(NODES)} capacity nodes | {len(ARCS)} arcs")

## 3. The subproblem, and its duals

In [ ]:
def subproblem(chat, k, get_duals=True):
    """Q_k(chat): recourse LP for scenario k. Returns (value, beta) with beta the capacity duals."""
    d = SC[k]
    m = gp.Model(); m.Params.OutputFlag = 0
    x = m.addVars(NODES, lb=0.0, name="x")
    f = m.addVars(ARCS,  lb=0.0, name="f")
    u = m.addVars(REGS,  lb=0.0, name="u")

    cap = m.addConstrs((x[n] <= chat[n] for n in NODES), name="cap")      # <-- the linking rows
    m.addConstrs((ETA[s]*x[s, r] == f.sum(s, r, "*") for (s, r) in NODES), name="out")
    for i, s in enumerate(STAGES):
        if i == 0:
            continue
        prev = STAGES[i-1]
        m.addConstrs((f.sum(prev, "*", r) == x[s, r] for r in REGS), name=f"in_{s}")
    m.addConstrs((f.sum("MFG", "*", r) + u[r] >= d[r] for r in REGS), name="dem")

    m.setObjective(gp.quicksum(OPC[s]*x[s, r] for (s, r) in NODES)
                   + gp.quicksum(TAU(r1, r2)*f[s, r1, r2] for (s, r1, r2) in ARCS)
                   + gp.quicksum(PEN*u[r] for r in REGS), GRB.MINIMIZE)
    m.optimize()
    assert m.Status == GRB.OPTIMAL, f"subproblem {k} not optimal: {m.Status}"
    beta = {n: cap[n].Pi for n in NODES} if get_duals else None
    return m.ObjVal, beta


## 4. The extensive form — the answer we must reproduce

Every scenario's second stage inside one model. This is the ground truth; L-shaped has to match it.

In [ ]:
def extensive_form(mipgap=0.0):
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    y = m.addVars(NODES, vtype=GRB.BINARY, name="y")
    c = m.addVars(NODES, lb=0.0, ub=CMAX, name="c")
    m.addConstrs((c[n] <= CMAX*y[n] for n in NODES))
    m.addConstrs((c[n] >= CMIN*y[n] for n in NODES))

    x = m.addVars(NODES, range(NK), lb=0.0)
    f = m.addVars(ARCS,  range(NK), lb=0.0)
    u = m.addVars(REGS,  range(NK), lb=0.0)
    for k in range(NK):
        m.addConstrs((x[s, r, k] <= c[s, r] for (s, r) in NODES))
        m.addConstrs((ETA[s]*x[s, r, k] == f.sum(s, r, "*", k) for (s, r) in NODES))
        for i, s in enumerate(STAGES):
            if i == 0: continue
            prev = STAGES[i-1]
            m.addConstrs((f.sum(prev, "*", r, k) == x[s, r, k] for r in REGS))
        m.addConstrs((f.sum("MFG", "*", r, k) + u[r, k] >= SC[k][r] for r in REGS))

    first  = gp.quicksum(FIX[s]*y[s, r] + UNIT[s]*c[s, r] for (s, r) in NODES)
    second = gp.quicksum(PK[k]*(gp.quicksum(OPC[s]*x[s, r, k] for (s, r) in NODES)
                                + gp.quicksum(TAU(r1, r2)*f[s, r1, r2, k] for (s, r1, r2) in ARCS)
                                + gp.quicksum(PEN*u[r, k] for r in REGS)) for k in range(NK))
    m.setObjective(first + second, GRB.MINIMIZE)
    m.optimize()
    return m, {n: c[n].X for n in NODES}

t0 = time.time()
ef, c_ef = extensive_form()
t_ef = time.time() - t0
print("extensive form: %.4f  (%.2fs, %d vars)" % (ef.ObjVal, t_ef, ef.NumVars))

## 5. The L-shaped loop

In [ ]:
def lshaped(max_iter=60, tol=1e-6, verbose=True):
    m = gp.Model(); m.Params.OutputFlag = 0
    y  = m.addVars(NODES, vtype=GRB.BINARY, name="y")
    c  = m.addVars(NODES, lb=0.0, ub=CMAX, name="c")
    th = m.addVars(range(NK), lb=0.0, name="theta")        # lb=0: costs are nonnegative
    m.addConstrs((c[n] <= CMAX*y[n] for n in NODES))
    m.addConstrs((c[n] >= CMIN*y[n] for n in NODES))
    first = gp.quicksum(FIX[s]*y[s, r] + UNIT[s]*c[s, r] for (s, r) in NODES)
    m.setObjective(first + gp.quicksum(PK[k]*th[k] for k in range(NK)), GRB.MINIMIZE)

    hist, UB = [], float("inf")
    for it in range(1, max_iter+1):
        m.optimize()
        LB   = m.ObjVal                       # master is a relaxation -> valid lower bound
        chat = {n: c[n].X for n in NODES}
        fc   = sum(FIX[s]*y[s, r].X + UNIT[s]*c[s, r].X for (s, r) in NODES)

        tot, ncuts = fc, 0
        for k in range(NK):
            Qk, beta = subproblem(chat, k)
            tot += PK[k]*Qk
            if th[k].X < Qk - tol*max(1.0, abs(Qk)):
                m.addConstr(th[k] >= Qk + gp.quicksum(beta[n]*(c[n] - chat[n]) for n in NODES))
                ncuts += 1
        UB = min(UB, tot)
        gap = (UB - LB)/max(1e-9, abs(UB))
        hist.append(dict(iter=it, LB=LB, UB=UB, gap=gap, cuts=ncuts))
        if verbose and (it <= 5 or it % 5 == 0 or ncuts == 0):
            print(f"  it {it:3d}  LB {LB:10.3f}  UB {UB:10.3f}  gap {100*gap:7.4f}%  cuts {ncuts}")
        if ncuts == 0 or gap < 1e-6:
            break
    return m, pd.DataFrame(hist), UB, {n: c[n].X for n in NODES}

t0 = time.time()
ls, hist, ls_val, c_ls = lshaped()
t_ls = time.time() - t0
print("\nL-shaped: %.4f  (%.2fs, %d iterations)" % (ls_val, t_ls, len(hist)))


## 6. Validation — it must match the extensive form

This is the assertion that matters. A Benders implementation with a sign error or a stale dual still
converges and still looks reasonable; only the comparison catches it.

In [ ]:
rel = abs(ls_val - ef.ObjVal)/abs(ef.ObjVal)
print("extensive form : %.6f" % ef.ObjVal)
print("L-shaped       : %.6f" % ls_val)
print("relative diff  : %.3e" % rel)
assert rel < 1e-5, "L-shaped did not reproduce the extensive form"
print("PASS\n")

cmp = pd.DataFrame([{"node": f"{s}/{r}", "EF": round(c_ef[s, r], 3), "L-shaped": round(c_ls[s, r], 3)}
                    for (s, r) in NODES])
print(cmp.to_string(index=False))
print("\n(capacity plans may differ where the objective is flat - the objective is what must match)")

## 7. Convergence

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["iter"], hist["LB"], "o-", label="lower bound (master)")
ax[0].plot(hist["iter"], hist["UB"], "s-", label="upper bound (incumbent)")
ax[0].axhline(ef.ObjVal, ls="--", c="k", lw=1, label="extensive form")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("objective"); ax[0].legend()
ax[0].set_title("Bounds close from both sides")

ax[1].semilogy(hist["iter"], hist["gap"].clip(lower=1e-12), "o-")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("relative gap")
ax[1].set_title("Gap is a certificate, not a guess")
plt.tight_layout(); plt.show()

print(hist.to_string(index=False))


### Single-cut, for comparison

Aggregating to one $\theta$ is a two-line change and converges much more slowly — each iteration
learns one hyperplane about the *average* recourse cost instead of $K$ about its parts.

In [ ]:
def lshaped_singlecut(max_iter=200, tol=1e-6):
    m = gp.Model(); m.Params.OutputFlag = 0
    y  = m.addVars(NODES, vtype=GRB.BINARY); c = m.addVars(NODES, lb=0.0, ub=CMAX)
    th = m.addVar(lb=0.0)
    m.addConstrs((c[n] <= CMAX*y[n] for n in NODES))
    m.addConstrs((c[n] >= CMIN*y[n] for n in NODES))
    first = gp.quicksum(FIX[s]*y[s, r] + UNIT[s]*c[s, r] for (s, r) in NODES)
    m.setObjective(first + th, GRB.MINIMIZE)
    UB = float("inf")
    for it in range(1, max_iter+1):
        m.optimize()
        LB = m.ObjVal; chat = {n: c[n].X for n in NODES}
        fc = sum(FIX[s]*y[s, r].X + UNIT[s]*c[s, r].X for (s, r) in NODES)
        Qbar, bbar = 0.0, {n: 0.0 for n in NODES}
        for k in range(NK):
            Qk, beta = subproblem(chat, k)
            Qbar += PK[k]*Qk
            for n in NODES: bbar[n] += PK[k]*beta[n]
        UB = min(UB, fc + Qbar)
        if th.X >= Qbar - tol*max(1.0, abs(Qbar)):
            break
        m.addConstr(th >= Qbar + gp.quicksum(bbar[n]*(c[n] - chat[n]) for n in NODES))
    return it, UB

it_sc, val_sc = lshaped_singlecut()
print("multicut : %3d iterations -> %.4f" % (len(hist), ls_val))
print("singlecut: %3d iterations -> %.4f" % (it_sc, val_sc))
print("\nsame optimum, %.1fx the iterations" % (it_sc/max(1, len(hist))))


## 8. Diagnostic — are the duals actually informative?

A cut whose coefficients are all zero tells the master nothing: capacity is not binding anywhere in
that scenario, so the hyperplane is flat and the iteration is wasted. If most $\beta$ are zero
early, the master is over-building and the instance is too loose to be interesting.

In [ ]:
chat0 = {n: 0.0 for n in NODES}
chatX = {n: c_ls[n] for n in NODES}
for label, ch in [("all-zero capacity", chat0), ("optimal capacity", chatX)]:
    nz, tot = 0, 0
    for k in range(NK):
        _, beta = subproblem(ch, k)
        nz  += sum(1 for n in NODES if abs(beta[n]) > 1e-9)
        tot += len(NODES)
    print(f"{label:20s}: {nz}/{tot} nonzero duals ({100*nz/tot:.0f}% binding)")

## 9. What this buys the production model

**The certificate.** Progressive hedging returns a plan; L-shaped returns a plan *and* a bound. For
a model that will inform investment recommendations, being able to say "within 0.4% of optimal" is
worth more than a slightly better incumbent.

**It scales in the direction the lithium problem grows.** Adding scenarios adds subproblems, which
are independent and embarrassingly parallel. Adding *periods* enlarges the master, which is where
this method is weakest — so if the production model grows mainly in scenarios, this is the right
tool; if it grows mainly in periods, revisit.

**Preconditions the series has already met:** integers confined to the first stage, complete
recourse via penalised shortfall, and a second stage that is a pure LP. Break any one of them and
the exactness goes away. In particular, a second stage containing binaries would leave $Q_k$
non-convex and the cuts invalid.

### Next

- Warm-start the master from the mean-value solution; the first few iterations are usually spent
  rediscovering it.
- Add a trust region on $c$ — the classic remedy for the early large steps visible in §7.
- For the risk-averse objective in Part 2c, the same decomposition applies with the CVaR epigraph in
  the master.